In [ ]:
!pip install scikit-learn
!pip install interpret

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 36.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.6/7.6 MB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.1/780.1 kB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 15.8 MB/s eta 0:00:00
  Created wheel for dash-cytoscape: filename=dash_cytoscape-1.0.2-py3-none-any.whl size=40

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
# from interpret.glassbox import ExplainableBoostingClassifier
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
import time
import pickle

In [10]:
base_path = "/content/drive/MyDrive/Fall 2025/Responsible AI/RAI Project/Project Implementation"
df = pd.read_csv(base_path + '/bisg_done_sample_760193.csv')
df_original = df.copy()

In [11]:
df.columns

Index(['first_name', 'last_name', 'middle_name', 'county_id', 'county_desc',
       'race_code', 'ethnic_code', 'zip_code', 'reason_cd', 'party_cd',
       'gender_code', 'age_at_year_end', 'drivers_lic', 'birth_state',
       'registr_dt', 'strata', 'zcta', 'prob_white', 'prob_black', 'prob_hisp',
       'prob_asian'],
      dtype='object')

In [12]:
df = df.drop(columns=["first_name", "middle_name", "last_name", 'zcta', 'prob_white', 'prob_black', 'prob_hisp',
       'prob_asian', 'race_code', 'ethnic_code' ], errors="ignore")
df["registr_year"] = pd.to_datetime(df["registr_dt"], errors="coerce").dt.year

target = "party_cd"

df = df[df[target].isin(["DEM", "REP", "UNA"])]

X = df.drop(columns=[target])
y = df[target]

In [13]:
df.columns

Index(['county_id', 'county_desc', 'zip_code', 'reason_cd', 'party_cd',
       'gender_code', 'age_at_year_end', 'drivers_lic', 'birth_state',
       'registr_dt', 'strata', 'registr_year'],
      dtype='object')

In [14]:
df_cleaned = df.dropna(subset=[target])
X = df_cleaned.drop(columns=[target])
y = df_cleaned[target]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [15]:
from sklearn.impute import SimpleImputer

categorical = [
    "county_id", "county_desc",
    "gender_code", "birth_state", "drivers_lic"
]
numeric = ["age_at_year_end", "registr_year"]


numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')) # Impute NaNs with the mean
])
preprocess = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical),
        ("num", numeric_transformer, numeric)
    ]
)

In [ ]:
# preprocess.fit(X_train)
# X_train_t = preprocess.transform(X_train)
# X_test_t  = preprocess.transform(X_test)

In [ ]:
# ebm = ExplainableBoostingClassifier(
#     interactions=0,
#     outer_bags=16,
#     learning_rate=0.01,
#     max_bins=256,
#     random_state=42
# )

# ebm = ExplainableBoostingClassifier(
#     interactions=0,
#     outer_bags=4,
#     inner_bags=1,
#     learning_rate=0.02,
#     max_bins=128,
#     random_state=42
# )

# start = time.time()
# ebm.fit(X_train_t, y_train)
# print("Training time:", time.time() - start, "seconds")

/usr/local/lib/python3.12/dist-packages/interpret/glassbox/_ebm/_ebm.py:872: UserWarning: Missing values detected. Our visualizations do not currently display missing values. To retain the glassbox nature of the model you need to either set the missing values to an extreme value like -1000 that will be visible on the graphs, or manually examine the missing value score in ebm.term_scores_[term_index][0]
  warn(


Training time: 2175.398138999939 seconds


In [ ]:
# with open("model.pkl", "wb") as f:
#     pickle.dump(ebm, f)

In [ ]:
# preds = ebm.predict(X_test_t)
# print(classification_report(y_test, preds))


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

         DEM       0.48      0.41      0.44     45197
         GRE       0.00      0.00      0.00        81
         LIB       0.00      0.00      0.00       902
         REP       0.47      0.40      0.43     43928
         UNA       0.49      0.61      0.54     57236

    accuracy                           0.48    147344
   macro avg       0.29      0.28      0.28    147344
weighted avg       0.48      0.48      0.48    147344



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [28]:
model = Pipeline([
    ("prep", preprocess),
    ("clf", RandomForestClassifier(
        n_estimators=10,
        random_state=42,
        n_jobs=-1,
        verbose = 2
    ))
])


model.fit(X_train, y_train)
with open("model_random_forest.pkl", "wb") as f:
    pickle.dump(model, f)

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 2 concurrent workers.


building tree 1 of 10building tree 2 of 10

building tree 3 of 10
building tree 4 of 10
building tree 5 of 10
building tree 6 of 10
building tree 7 of 10
building tree 8 of 10
building tree 9 of 10
building tree 10 of 10


[Parallel(n_jobs=-1)]: Done  10 out of  10 | elapsed: 10.1min finished


In [29]:
preds = model.predict(X_test)
print(classification_report(y_test, preds))

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    0.9s finished


              precision    recall  f1-score   support

         DEM       0.41      0.40      0.41     45197
         REP       0.40      0.38      0.39     43928
         UNA       0.46      0.48      0.47     57236

    accuracy                           0.43    146361
   macro avg       0.42      0.42      0.42    146361
weighted avg       0.42      0.43      0.43    146361



In [24]:
df_orignal = df_original.dropna(subset=[target])

In [30]:
predictions = model.predict(X)
predicted_party_series = pd.Series(predictions, index=X.index)
df_original["pred_party"] = predicted_party_series
df_original.to_csv("nc_voted_with_predicted_party.csv")

[Parallel(n_jobs=2)]: Using backend ThreadingBackend with 2 concurrent workers.
[Parallel(n_jobs=2)]: Done  10 out of  10 | elapsed:    4.9s finished
